# Vélib Real-Time Station Data - Silver Layer

## Objective
Transform, cleanse, and standardize raw Vélib bicycle station data from bronze.bronze_velib to build a clean, production-ready Silver Delta table (silver.silver_velib).

## Data Flow
bronze.bronze_velib → Spark DataFrame → silver.silver_velib

## Source
The underlying data comes from the Paris OpenData API: Vélib - Emplacement des stations - Disponibilité en temps réel.

## Input
Bronze Delta table: bronze.bronze_velib

## Output
Silver Delta table: silver.silver_velib

## Silver Layer Principle
The Silver layer cleanses, enforces schema integrity, and standardizes data structures for downstream analytics. Column names are mapped from French/technical identifiers to standardized English snake_case names, date/time fields are cast to appropriate temporal types, and null value distributions are audited prior to persistence.

## Processing Steps
1. **Load Bronze Data:** Read raw table into PySpark and verify row count.
2. **Standardize Column Names:** Rename columns to English snake_case.
3. **Data Type Conversions:** Cast duedate string to TimestampType.
4. **Data Quality & Null Auditing:** Count null values across all columns.
5. **Write to Silver:** Save cleaned DataFrame to silver.silver_velib Delta table.

In [0]:
# Importing libraries
from pyspark.sql.functions import col, count, when
from pyspark.sql import functions as F

# LOAD BRONZE DATA

In [0]:
# Load data from bronze schema
df_bronze_velib=spark.table("workspace.bronze.bronze_velib")

In [0]:
# Inspecting the schema 
df_bronze_velib.printSchema()

In [0]:
# count number of rows in `df_bronze_velib` 
df_count=df_bronze_velib.count()
print(df_count)

In [0]:
# display the dataframe 
df_bronze_velib.limit(10).display()

# CLEAN DATA  AND CHECK FOR NULLS

In [0]:
# Mapping raw Vélib column names to standardized English snake_case names
velib_column_mapping = {
    "stationcode": "station_code",
    "name": "station_name",
    "is_installed": "is_installed",
    "capacity": "capacity",
    "numdocksavailable": "num_docks_available",
    "numbikesavailable": "num_bikes_available",
    "mechanical": "num_mechanical_bikes",
    "ebike": "num_ebikes",
    "is_renting": "is_renting",
    "is_returning": "is_returning",
    "duedate": "due_date",
    "coordonnees_geo": "geo_coordinates",
    "nom_arrondissement_communes": "district_commune_name",
    "code_insee_commune": "insee_commune_code",
    "station_opening_hours": "station_opening_hours",
    "_ingestion_timestamp": "_ingestion_timestamp"
}

# Loop to dynamically rename columns in your PySpark DataFrame
for old_col, new_col in velib_column_mapping.items():
    df_velib = df_bronze_velib.withColumnRenamed(old_col, new_col)

In [0]:
# Display the new column names 
df_bronze_velib.limit(10).display()

In [0]:
# Convert French string indicators to native Boolean flags
df_clean = (
    df_bronze_velib
    .withColumn("is_installed", F.when(F.upper(F.trim(col("is_installed"))) == "OUI", True).otherwise(False))
    .withColumn("is_renting", F.when(F.upper(F.trim(col("is_renting"))) == "OUI", True).otherwise(False))
    .withColumn("is_returning", F.when(F.upper(F.trim(col("is_returning"))) == "OUI", True).otherwise(False))
)

In [0]:
# Display the new dataframe
df_clean.limit(10).display()

In [0]:
# Cast duedate to timestamp 
df_clean = df_clean.withColumn(
    "duedate",
    F.to_timestamp(F.col("duedate"), "yyyy-MM-dd'T'HH:mm:ssXXX")
)
df_clean.limit(10).display()

In [0]:
# Check for nulls
null_counts_df = df_clean.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_clean.columns
])

# Display the summary table showing NULL count per column
display(null_counts_df)

# WRITE TO SILVER LAYER

In [0]:
df_clean\
    .write\
        .format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true") \
        .saveAsTable("silver.silver_velib")

# CHECKING THE SILVER TABLE

In [0]:
%sql 
SELECT *
FROM workspace.silver.silver_velib;